### SAT as CSP

In [6]:
from mermaid import Mermaid
print(Mermaid("constraint graph:\n"))

graph = Mermaid("""
graph LR
X1 --- X2
X2 --- X3
X3 --- X4
X4 --- X5
""")
graph

In [8]:
# Import CSP and search modules
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from csp import CSP
from search4e import Node

In [ ]:
# Define the implication chain constraint function
# Constraint: (¬Xi ∨ Xi+1) which is equivalent to (¬Xi OR Xi+1) or (Xi => Xi+1)

def implication_constraint(A, a, B, b):
    """
    Check if assignment satisfies (¬A OR B).
    This is equivalent to: A => B, or NOT(A AND NOT B)
    
    Args:
        A, B: variable names
        a, b: assigned values (True/False)
    
    Returns:
        True if constraint is satisfied, False otherwise
    """
    # (¬A ∨ B) is violated only when A=True and B=False
    return not (a and not b)


def create_implication_chain_csp(n):
    """
    Create a CSP for the implication chain: (¬X1 OR X2) ∧ (¬X2 OR X3) ∧ ... ∧ (¬Xn-1 OR Xn)
    
    Args:
        n: number of variables
    
    Returns:
        CSP problem instance
    """
    # Variable names: X1, X2, ..., Xn
    variables = [f'X{i}' for i in range(1, n + 1)]
    
    # All variables have binary domain: {True, False}
    domains = {var: [True, False] for var in variables}
    
    # Build neighbors dictionary: Xi is neighbor of Xi+1 (and vice versa)
    neighbors = {var: [] for var in variables}
    for i in range(n - 1):
        neighbors[variables[i]].append(variables[i + 1])
        neighbors[variables[i + 1]].append(variables[i])
    
    # Create CSP with implication constraint
    csp = CSP(variables, domains, neighbors, implication_constraint)
    return csp

# Example: Create CSP with n=5 variables
n = 5
csp_problem = create_implication_chain_csp(n)

print(f"Created CSP with {len(csp_problem.variables)} variables")
print(f"Variables: {csp_problem.variables}")
print(f"Domains: Binary {{True, False}}")
print(f"Constraints: (¬Xi ∨ Xi+1) for all consecutive pairs")

Created CSP with 5 variables
Variables: ['X1', 'X2', 'X3', 'X4', 'X5']
Domains: Binary {True, False}
Constraints: (¬Xi ∨ Xi+1) for all consecutive pairs


In [10]:
# Option B: Modified recursive backtracking to collect ALL solutions

def find_all_solutions(csp, assignment=None, solutions=None):
    """
    Find ALL solutions to a CSP using modified backtracking search.
    Collects solutions before backtracking (instead of returning on first solution).
    
    Args:
        csp: CSP problem instance
        assignment: current partial assignment (dict)
        solutions: list to collect all solutions
    
    Returns:
        List of all valid complete assignments
    """
    if assignment is None:
        assignment = {}
    if solutions is None:
        solutions = []
    
    # Base case: all variables assigned
    if len(assignment) == len(csp.variables):
        # Found a complete solution
        solutions.append(assignment.copy())
        return solutions
    
    # Select unassigned variable (simple: first unassigned)
    unassigned = [var for var in csp.variables if var not in assignment]
    var = unassigned[0]
    
    # Try each value in the domain
    for value in csp.domains[var]:
        # Check if assignment is valid (no conflicts with neighbors)
        if is_consistent(var, value, assignment, csp):
            # Assign value
            assignment[var] = value
            
            # Recursively search for solutions
            find_all_solutions(csp, assignment, solutions)
            
            # Backtrack: unassign value (key: continues to other values)
            del assignment[var]
    
    return solutions


def is_consistent(var, value, assignment, csp):
    """
    Check if assigning var=value is consistent with current assignment.
    Verifies all constraints between var and already-assigned neighbors.
    
    Args:
        var: variable name
        value: proposed value
        assignment: current partial assignment
        csp: CSP problem
    
    Returns:
        True if assignment is consistent, False otherwise
    """
    for neighbor in csp.neighbors[var]:
        if neighbor in assignment:
            # Both var and neighbor are assigned, check constraint
            if not csp.constraints(var, value, neighbor, assignment[neighbor]):
                return False
    return True

print("Defined find_all_solutions() using modified backtracking (Option B)")
print("Key feature: Collects all solutions before each return statement")

Defined find_all_solutions() using modified backtracking (Option B)
Key feature: Collects all solutions before each return statement


In [16]:
# Run the search and collect all solutions
import time

print("=" * 70)
print(f"Finding ALL solutions for implication chain (n={n})")
print("=" * 70)
print(f"Constraint: (NOT X1 OR X2) ∧ (NOT X2 OR X3) ∧ ... ∧ (NOT X{n-1} OR X{n})")
print()

start_time = time.time()
all_solutions = find_all_solutions(csp_problem) # run the backtrack search
elapsed_time = time.time() - start_time

print(f"Time to find all solutions: {elapsed_time:.6f} seconds")
print(f"Total solutions found: {len(all_solutions)}")
print(f"Expected (n+1): {n + 1}")
print()

# Display all solutions
print("All solutions:")
print("-" * 70)
for i, solution in enumerate(all_solutions, 1):
    values = [solution[f'X{j}'] for j in range(1, n + 1)]
    bool_str = ''.join('T' if v else 'F' for v in values)
    num_true = sum(values)
    print(f"  {i:2d}. {bool_str}  (#{num_true} True values)")

print("-" * 70)

Finding ALL solutions for implication chain (n=5)
Constraint: (NOT X1 OR X2) ∧ (NOT X2 OR X3) ∧ ... ∧ (NOT X4 OR X5)

Time to find all solutions: 0.000101 seconds
Total solutions found: 6
Expected (n+1): 6

All solutions:
----------------------------------------------------------------------
   1. TTTTT  (#5 True values)
   2. TTTTF  (#4 True values)
   3. TTTFF  (#3 True values)
   4. TTFFF  (#2 True values)
   5. TFFFF  (#1 True values)
   6. FFFFF  (#0 True values)
----------------------------------------------------------------------


In [14]:
# Verify all solutions satisfy the constraints

def verify_solution(solution, n):
    """
    Verify that a solution satisfies all implication chain constraints.
    Constraint: (¬Xi ∨ Xi+1) for all i = 1, 2, ..., n-1
    """
    for i in range(1, n):
        xi = solution[f'X{i}']
        xi_next = solution[f'X{i+1}']

        # Check constraint: NOT(xi AND NOT xi_next)
        constraint_satisfied = not (xi and not xi_next)
        if not constraint_satisfied:
            return False, f"Constraint (NOT X{i} OR X{i+1}) violated: X{i}={xi}, X{i+1}={xi_next}"
    return True, "All constraints satisfied"

print("Verification of all solutions:")
print("-" * 70)

all_valid = True

for i, solution in enumerate(all_solutions, 1):
    valid, msg = verify_solution(solution, n)
    status = "VALID" if valid else "INVALID"
    print(f"  Solution {i:2d}: {status}")
    if not valid:
        print(f"    {msg}")
        all_valid = False

print("-" * 70)
if all_valid:
    print("All solutions are VALID and satisfy constraints!")
else:
    print("Some solutions are INVALID!")

Verification of all solutions:
----------------------------------------------------------------------
  Solution  1: VALID
  Solution  2: INVALID
    Constraint (NOT X4 OR X5) violated: X4=True, X5=False
  Solution  3: INVALID
    Constraint (NOT X3 OR X4) violated: X3=True, X4=False
  Solution  4: INVALID
    Constraint (NOT X2 OR X3) violated: X2=True, X3=False
  Solution  5: INVALID
    Constraint (NOT X1 OR X2) violated: X1=True, X2=False
  Solution  6: VALID
----------------------------------------------------------------------
Some solutions are INVALID!


### Symbols

In [1]:
from pathlib import Path
import sys
# set current cwd parent to path
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from logic4e import pl_true, tt_entails, tt_entails_traced, trace_dpll, dpll_satisfiable_traced
from utils import symbols, expr
import time

In [2]:
# create model
P, Q, R = symbols('P Q R')
mod = {P: False, Q: True}

# create expression
e1 = expr('P & R') # and defines missing R
e2 = expr('P ==> R')  # implication defines missing R
e3 = expr('Q <== R')  # reverse implication defines missing R

expresssions = [e1, e2, e3]
# evaluate expressions in model
results = [pl_true(e, mod) for e in expresssions]
results

[False, True, True]

In [61]:
# time the evaluation

st1 = time.time()
e1 = expr('P')
et1 = time.time()
print(f"e1 took {et1 - st1}")

st2 = time.time()
e2 = expr('((P | R) & (P ==> R)) <=> (P | (R & P)) ==> (R | P) ==> (R | (P & R))')
et2 = time.time()
print(f"e2 took {et2 - st2}")

e1 took 0.00012969970703125
e2 took 0.00023627281188964844


In [7]:
# Creating fresh symbols for tracing demonstration

from utils4e import symbols as utils_symbols

# Create fresh symbols for tracing
A, B, C = utils_symbols('A B C')

print("="*80)
print("TRACING: Does (A & B) entail B?")
result_traced_1 = tt_entails_traced(A & B, B)

print("="*80)
print("TRACING: Does (A | B) entail B?")
result_traced_2 = tt_entails_traced(A | B, B)

TRACING: Does (A & B) entail B?

[STARTING tt_entails_traced]
├─ KB: (A & B)
├─ Alpha (Query): B
├─ All Symbols to Process: [B, A]
└─ Initial Model: {}


[CALL #1] tt_check_all()
├─ Recursion Depth: 1
├─ Model: {}
├─ Remaining Symbols: [B, A]
├─ KB Expression: (A & B)
└─ Alpha Query: B

[RECURSIVE CASE] Processing symbol: B
├─ Next Symbol to Branch: B
├─ Remaining Symbols after B: [A]

[BRANCH 1] Setting B = True
├─ Extended Model: {B: True}

  [CALL #2] tt_check_all()
  ├─ Recursion Depth: 2
  ├─ Model: {B: True}
  ├─ Remaining Symbols: [A]
  ├─ KB Expression: (A & B)
  └─ Alpha Query: B
  
[RECURSIVE CASE] Processing symbol: A
  ├─ Next Symbol to Branch: A
  ├─ Remaining Symbols after A: []
  
  [BRANCH 1] Setting A = True
  ├─ Extended Model: {B: True, A: True}

    [CALL #3] tt_check_all()
    ├─ Recursion Depth: 3
    ├─ Model: {B: True, A: True}
    ├─ Remaining Symbols: []
    ├─ KB Expression: (A & B)
    └─ Alpha Query: B
    
[BASE CASE] No more symbols to assign
    ├─ Compl

### How `HybridWumpusAgent` works
- **Hybrid design**: combines propositional/FOL logical inference (a KB) with reactive behaviors.
- **Percepts → KB**: the agent receives percepts (stench, breeze, bump, glitter, scream) and updates its knowledge base.
- **Logical reasoning**: it uses the KB (forward/backward chaining, SAT/DPLL, or other inference routines in `logic4e`) to deduce safe squares, possible wumpus/pit locations, and goals.
- **Action selection**: the agent picks actions by querying the KB (e.g., is it safe to move forward?) and combining that with short-term plans or default behaviors.
- **Integration**: this agent is useful for environments where both symbolic reasoning and quick reactions are needed: reasoning increases safety and goal achievement, while reactive code handles immediate responses.

In [6]:
# Create a HybridWumpusAgent instance from logic4e
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent.parent.resolve()))

from logic4e import HybridWumpusAgent

agent = HybridWumpusAgent(4)
print('Created HybridWumpusAgent:', type(agent).__name__)

Created HybridWumpusAgent: HybridWumpusAgent
